In [1]:
import numpy as np
import random


In [2]:
# ==========================================
# 1. ENVIRONMENT SETUP
# ==========================================
GRID_SIZE = 3
START_STATE = (0, 0) # Top-left corner
GOAL_STATE = (2, 2)  # Bottom-right corner

# Actions: 0=Up, 1=Right, 2=Down, 3=Left
NUM_ACTIONS = 4

# Initialize the Q-Table with zeros. 
# It is a 3D array: 3x3 grid, and 4 possible actions for each cell.
q_table = np.zeros((GRID_SIZE, GRID_SIZE, NUM_ACTIONS))

In [3]:
# ==========================================
# 2. HYPERPARAMETERS
# ==========================================
ALPHA = 0.1    # Learning Rate: How much new info overrides old info
GAMMA = 0.9    # Discount Factor: How much we care about future rewards (causes the ripple!)
EPSILON = 0.1  # Exploration Rate: 10% of the time, take a random move
EPISODES = 500 # How many times the agent will play the game

In [4]:
# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def get_next_state(state, action):
    """Calculates the next position based on the chosen action."""
    r, c = state
    if action == 0 and r > 0:               r -= 1  # Move Up
    elif action == 1 and c < GRID_SIZE - 1: c += 1  # Move Right
    elif action == 2 and r < GRID_SIZE - 1: r += 1  # Move Down
    elif action == 3 and c > 0:             c -= 1  # Move Left
    # Note: If an action pushes the agent into a wall, it just stays in the same cell.
    return (r, c)

In [5]:
def get_reward(state):
    """Returns the reward for stepping into a state."""
    if state == GOAL_STATE:
        return 10  # The big payload!
    return -1      # The step penalty (encourages finding the shortest path)

In [6]:
# ==========================================
# 4. THE Q-LEARNING ALGORITHM (TRAINING LOOP)
# ==========================================
for episode in range(EPISODES):
    state = START_STATE
    
    # Keep stepping until the agent reaches the goal
    while state != GOAL_STATE:
        
        # --- A. EPSILON-GREEDY POLICY (The Agent's Choice) ---
        if random.uniform(0, 1) < EPSILON:
            # Explore: Pick a completely random action
            action = random.choice([0, 1, 2, 3])
        else:
            # Exploit: Look at the Q-table and pick the best known action for this cell
            action = np.argmax(q_table[state[0], state[1]])
            
        # --- B. TAKE THE STEP ---
        next_state = get_next_state(state, action)
        reward = get_reward(next_state)
        
        # --- C. Q-TABLE UPDATE (The Ripple Effect / Bootstrapping) ---
        # 1. What was our old score for this move?
        old_value = q_table[state[0], state[1], action]
        
        # 2. What is the BEST possible score we can get from the NEXT cell? (Bootstrapping)
        next_max = np.max(q_table[next_state[0], next_state[1]])
        
        # 3. The Q-Learning Formula (Bellman Equation)
        # We combine the immediate reward (-1) with the estimated future potential (GAMMA * next_max)
        new_value = old_value + ALPHA * (reward + (GAMMA * next_max) - old_value)
        
        # 4. Save the new score in the cheat sheet
        q_table[state[0], state[1], action] = new_value
        
        # --- D. MOVE TO THE NEXT STATE ---
        state = next_state

In [7]:
# ==========================================
# 5. VISUALIZING THE RESULTS
# ==========================================
print("Training Complete! Here is the Agent's Map (The Ripple Effect):")
print("-" * 50)

actions_symbols = ['↑', '→', '↓', '←']

for r in range(GRID_SIZE):
    row_visual = []
    for c in range(GRID_SIZE):
        if (r, c) == GOAL_STATE:
            row_visual.append("[ GOAL ]")
        else:
            # Find the best action for this cell
            best_action = np.argmax(q_table[r, c])
            best_value = np.max(q_table[r, c])
            
            # Format to show the arrow and the Q-value (e.g., "→: +7.1")
            row_visual.append(f"[{actions_symbols[best_action]}: {best_value:+.1f}]")
            
    print("  ".join(row_visual))
    print("")

print("-" * 50)
print("Notice how the values get higher (+8.0, +9.0) the closer you get to the GOAL.")
print("The agent simply follows the highest numbers from the Start (0,0) to the Goal!")

Training Complete! Here is the Agent's Map (The Ripple Effect):
--------------------------------------------------
[↓: +4.6]  [↓: +5.9]  [↓: +4.3]

[→: +6.2]  [→: +8.0]  [↓: +10.0]

[→: +3.7]  [→: +9.2]  [ GOAL ]

--------------------------------------------------
Notice how the values get higher (+8.0, +9.0) the closer you get to the GOAL.
The agent simply follows the highest numbers from the Start (0,0) to the Goal!
